# 01 · Single-checkpoint probe

**Grand Challenge Labs · Coupling-Phase Spectroscopy**

Measure the local projected optimizer-state dynamics of Pythia-70M and learn how to read the resulting operator, coupling table, and evidence manifest.


## Release contract

| Contract | Declared value |
|---|---|
| **Scientific question** | What local coupling structure does CPS measure at one compact Pythia-70M checkpoint? |
| **Default path** | Run the governed smoke configuration with reconstructed Adam moments and a compact semantic basis. |
| **Evidence boundary** | This validates the real-model measurement path. Reconstructed moments are not a claim about historical native optimizer state. |
| **Primary outputs** | Manifest, basis registry, reduced operator, ranked couplings, figures, and export archive. |


## Interpretation checklist

- [ ] Read projection closure before interpreting spectral radius or transient gain.
- [ ] Verify the active attention and JVP backends recorded in the manifest.
- [ ] Treat strong couplings as diagnostic leads, not as interventions or causal findings.


## What this run establishes

- It validates a real Pythia forward pass inside a differentiable selected-coordinate AdamW map.
- It constructs a small semantic projection rather than materializing the full optimizer-state Jacobian.
- It measures phase sensitivity of the strongest directed couplings in that projection.
- It writes the exact attention and JVP backends into the evidence manifest.

**Evidence boundary.** The default smoke configuration reconstructs moments rather than loading historical Adam state. It validates the instrument on a real model, not a historical claim about step 0 dynamics.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import apply_release_theme, stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
apply_release_theme()
runtime = show_environment()

## Why the fused-SDPA error occurred

`torch.func.jvp` uses forward-mode automatic differentiation. PyTorch's fused `_scaled_dot_product_efficient_attention` kernel currently lacks the required forward-AD rule, even though its ordinary forward and reverse-mode backward passes work.

CPS now handles this in two layers:

1. the Pythia probe requests the **eager attention implementation**, whose elementary matrix operations admit forward AD;
2. a JVP preflight runs before projection. If any active kernel still lacks forward AD, CPS switches to a declared centered finite-difference JVP and records the fallback and exception in `manifest.json`.

Thus the notebook does not silently change methods, and it does not fail halfway through the projection.

## Stage 1 — inspect the experiment contract

The smoke run probes one first-layer attention output matrix, rank four, over 33 phase samples. Small rank is intentional: it lets us validate the full path before increasing projection breadth.

In [ ]:
from cps.notebook import stage_banner
stage_banner('1', 'inspect the experiment contract', objective='The smoke run probes one first-layer attention output matrix, rank four, over 33 phase samples. Small rank is intentional: it lets us validate the full path before increasing projection breadth.', deliverable="A verified stage result recorded in the evidence packet.")

from cps.notebook import show_config
from cps.pythia.config import load_probe_config

config = load_probe_config("subjects/pythia/configs/pythia_70m_smoke.yaml")
contract = show_config(config)

## Stage 2 — execute the probe

Watch for the following milestones in the live log:

- active attention backend;
- selected tensor and state dimension;
- semantic basis labels;
- effective JVP backend after preflight;
- closure residual for every projected column;
- phase-envelope spectral radius for each selected coupling.

In [ ]:
from cps.notebook import stage_banner
stage_banner('2', 'execute the probe', objective='Watch for the following milestones in the live log:', deliverable="A verified stage result recorded in the evidence packet.")

from cps.pythia.runner import run_probe

output = run_probe(config)
print(f"\n[NOTEBOOK] Probe evidence root: {output}", flush=True)

## Stage 3 — read the result as a diagnostic, not merely a file

The first figure is the magnitude of the reduced operator. Rows are target modes; columns are source modes. The second ranks directed couplings by their worst spectral radius over phase.

In [ ]:
from cps.notebook import stage_banner
stage_banner('3', 'read the result as a diagnostic, not merely a file', objective='The first figure is the magnitude of the reduced operator. Rows are target modes; columns are source modes. The second ranks directed couplings by their worst spectral radius over phase.', deliverable="A verified stage result recorded in the evidence packet.")

from cps.notebook import display_probe_summary

summary = display_probe_summary(output, top_n=8)

## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
